# DDoS Attack Detection Using LSTM-Based Network Traffic Analysis

This notebook walks through the complete pipeline for detecting Distributed Denial-of-Service (DDoS) attacks using a Bidirectional LSTM with temporal attention.

**Dataset:** CIC-DDoS2019 / CICIDS-2017 (or synthetic data generated by `scripts/generate_synthetic_data.py`)

**Pipeline overview:**
1. Load & explore → 2. Preprocess → 3. Scale → 4. Build sequences → 5. Train LSTM → 6. Evaluate → 7. Real-time simulation

## 1  Import Required Libraries

In [1]:
import sys, warnings, os
from pathlib import Path
warnings.filterwarnings("ignore")

# Allow importing from repo root
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve,
    roc_auc_score, precision_recall_curve, average_precision_score,
)

# Project modules
import config as cfg
from src.model   import build_model
from src.utils   import set_seed, get_device, compute_class_weights, predict

# Reproducibility
set_seed(cfg.RANDOM_STATE)
DEVICE = get_device()

print(f"PyTorch {torch.__version__}  |  Device: {DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

ModuleNotFoundError: No module named 'seaborn'

## 2  Load and Explore the Dataset

If you have the CIC-DDoS2019 CSV, place it at `data/raw/network_traffic.csv`.  
Otherwise, run the cell below to generate a **synthetic** dataset with the same schema.

In [ ]:
# ── Generate synthetic data if real CSV is absent ──────────────────────────
if not cfg.RAW_CSV.exists():
    print("Real dataset not found – generating synthetic data …")
    from scripts.generate_synthetic_data import generate
    generate()

# ── Load CSV ────────────────────────────────────────────────────────────────
df_raw = pd.read_csv(cfg.RAW_CSV, low_memory=False)
df_raw.columns = df_raw.columns.str.strip()
print(f"Shape  : {df_raw.shape}")
print(f"Memory : {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df_raw.head(3)

In [ ]:
# ── Class distribution ──────────────────────────────────────────────────────
label_col = "Label"
counts    = df_raw[label_col].value_counts()
print("Class distribution:")
print(counts.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts.plot(kind="bar", ax=axes[0], color=sns.color_palette("muted", len(counts)))
axes[0].set_title("Sample Counts per Class")
axes[0].set_xlabel("")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=30)

# Pie chart
axes[1].pie(counts, labels=counts.index, autopct="%1.1f%%",
            colors=sns.color_palette("muted", len(counts)))
axes[1].set_title("Class Proportion")
plt.tight_layout()
plt.show()

In [ ]:
# ── Missing-value & basic stats ─────────────────────────────────────────────
print("Missing values:", df_raw.isnull().sum().sum())
print(f"\nDtypes:\n{df_raw.dtypes.value_counts()}")
print(f"\nDescriptive stats (first 5 numeric features):")
numeric_cols = df_raw.select_dtypes(include=np.number).columns
df_raw[numeric_cols[:5]].describe().round(2)

## 3  Data Preprocessing and Feature Engineering

In [ ]:
from src.preprocess import (
    load_and_clean, select_features, encode_labels,
    split_data, fit_scaler, scale, make_sequences, oversample_sequences,
)

# ── Clean ────────────────────────────────────────────────────────────────────
df = load_and_clean(cfg.RAW_CSV)

# ── Feature correlation heatmap (top 20 features) ───────────────────────────
num_cols = df.select_dtypes(include=np.number).columns[:20]
fig, ax  = plt.subplots(figsize=(14, 11))
corr     = df[num_cols].corr()
mask     = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0,
            linewidths=0.3, annot=False, ax=ax)
ax.set_title("Feature Correlation Matrix (top-20 features)")
plt.tight_layout()
plt.show()

In [ ]:
# ── Select features & encode labels ─────────────────────────────────────────
X, feature_names = select_features(df)
y, le            = encode_labels(df, binary=True)

print(f"Feature matrix : {X.shape}")
print(f"Labels         : {np.unique(y, return_counts=True)}")
print(f"Classes        : {le.classes_}")

# ── Feature importance (variance-based proxy) ────────────────────────────────
var      = np.var(X, axis=0)
top_idx  = np.argsort(var)[::-1][:20]
top_names = [feature_names[i] for i in top_idx]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_names[::-1], var[top_idx[::-1]], color="steelblue")
ax.set_xlabel("Variance")
ax.set_title("Top-20 Features by Variance")
plt.tight_layout()
plt.show()

## 4  Normalize and Scale Features

In [ ]:
# ── Train / Val / Test split ──────────────────────────────────────────────────
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

# ── RobustScaler (insensitive to outliers) ────────────────────────────────────
scaler = fit_scaler(X_train)
X_train_sc, X_val_sc, X_test_sc = scale(scaler, X_train, X_val, X_test)

# ── Visualise scaling effect ──────────────────────────────────────────────────
feat_idx = 0          # show effect for first feature
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, title in zip(
    axes,
    [X_train[:, feat_idx], X_train_sc[:, feat_idx]],
    ["Before Scaling", "After RobustScaler"],
):
    ax.hist(data, bins=60, color="steelblue", edgecolor="none", alpha=0.8)
    ax.set_title(f"{title} — {feature_names[feat_idx]}")
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()
print(f"Train scaled  mean={X_train_sc.mean():.4f}  std={X_train_sc.std():.4f}")

## 5  Sequence Generation for LSTM Input

In [ ]:
SEQ_LEN = cfg.SEQUENCE_LEN
print(f"Sliding window length: {SEQ_LEN}")

X_train_s, y_train_s = make_sequences(X_train_sc, y_train, SEQ_LEN)
X_val_s,   y_val_s   = make_sequences(X_val_sc,   y_val,   SEQ_LEN)
X_test_s,  y_test_s  = make_sequences(X_test_sc,  y_test,  SEQ_LEN)

print(f"Train sequences : {X_train_s.shape}  labels: {y_train_s.shape}")
print(f"Val  sequences  : {X_val_s.shape}")
print(f"Test sequences  : {X_test_s.shape}")

# ── Visualise a sample sequence ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
sample  = X_train_s[0]                        # (SEQ_LEN, n_features)
im      = ax.imshow(sample.T, aspect="auto", cmap="viridis", interpolation="nearest")
ax.set_xlabel("Timestep")
ax.set_ylabel("Feature index")
ax.set_title(f"Sample LSTM Input  |  Label: {le.classes_[y_train_s[0]]}  "
             f"|  Shape: {sample.shape}")
plt.colorbar(im, ax=ax, label="Scaled value")
plt.tight_layout()
plt.show()

## 6  Train / Test Split (already done above — confirmation cell)

In [ ]:
# Optional SMOTE oversampling
if cfg.OVERSAMPLE:
    X_train_s, y_train_s = oversample_sequences(X_train_s, y_train_s)

# ── Split summary ─────────────────────────────────────────────────────────────
for name, yt in zip(["Train", "Val", "Test"], [y_train_s, y_val_s, y_test_s]):
    u, c = np.unique(yt, return_counts=True)
    print(f"{name:5s}: total={len(yt):6d}  "
          + "  ".join(f"{le.classes_[ui]}={ci}" for ui, ci in zip(u, c)))

## 7  Build the LSTM Model

In [ ]:
n_features = X_train_s.shape[2]
n_classes  = len(le.classes_)

model = build_model(n_features, n_classes).to(DEVICE)
print(model)
print(f"\nTotal trainable parameters: {model.count_parameters():,}")

## 8  Compile and Train the Model

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR
from src.utils import compute_class_weights, save_checkpoint, load_checkpoint, format_elapsed
import time

# ── Data loaders ──────────────────────────────────────────────────────────────
def make_loader(X, y, shuffle=False, bs=cfg.BATCH_SIZE):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=0, pin_memory=True)

train_loader = make_loader(X_train_s, y_train_s, shuffle=True)
val_loader   = make_loader(X_val_s,   y_val_s)
test_loader  = make_loader(X_test_s,  y_test_s)

# ── Optimiser & loss ──────────────────────────────────────────────────────────
weights   = compute_class_weights(y_train_s, n_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = AdamW(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)
scheduler = StepLR(optimizer, step_size=cfg.LR_STEP_SIZE, gamma=cfg.LR_GAMMA)
amp_scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

# ── History ───────────────────────────────────────────────────────────────────
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss = float("inf")
patience_counter = 0
best_ckpt = cfg.MODELS_DIR / "nb_best.pt"

EPOCHS = cfg.EPOCHS
print(f"Starting training for up to {EPOCHS} epochs …")
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
            out  = model(Xb)
            loss = criterion(out, yb)
        optimizer.zero_grad(set_to_none=True)
        amp_scaler.scale(loss).backward()
        amp_scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
        amp_scaler.step(optimizer)
        amp_scaler.update()
        t_loss    += loss.item() * yb.size(0)
        t_correct += (out.argmax(1) == yb).sum().item()
        t_total   += yb.size(0)

    # ── Val ──
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            out    = model(Xb)
            loss   = criterion(out, yb)
            v_loss    += loss.item() * yb.size(0)
            v_correct += (out.argmax(1) == yb).sum().item()
            v_total   += yb.size(0)

    tl = t_loss / t_total;  ta = t_correct / t_total
    vl = v_loss / v_total;  va = v_correct / v_total
    history["train_loss"].append(tl); history["train_acc"].append(ta)
    history["val_loss"].append(vl);   history["val_acc"].append(va)
    scheduler.step()

    print(f"Ep {epoch:3d}/{EPOCHS}  train={tl:.4f}/{ta:.4f}  "
          f"val={vl:.4f}/{va:.4f}  [{format_elapsed(t0)}]")

    if vl < best_val_loss:
        best_val_loss = vl
        patience_counter = 0
        save_checkpoint(model, optimizer, epoch, vl, best_ckpt)
    else:
        patience_counter += 1
        if patience_counter >= cfg.PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

print("Training complete.")
load_checkpoint(model, best_ckpt, DEVICE)

## 9  Evaluate Model Performance

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

# ── Inference ─────────────────────────────────────────────────────────────────
all_probs, all_preds, all_labels = predict(model, test_loader, DEVICE)
class_names = list(le.classes_)

# ── Scalar metrics ─────────────────────────────────────────────────────────────
acc   = accuracy_score(all_labels, all_preds)
f1_m  = f1_score(all_labels, all_preds, average="macro",    zero_division=0)
f1_w  = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
kappa = cohen_kappa_score(all_labels, all_preds)
auc   = roc_auc_score(all_labels, all_probs[:, 1])

print(f"{'Accuracy':<25}: {acc:.4f}")
print(f"{'F1 (macro)':<25}: {f1_m:.4f}")
print(f"{'F1 (weighted)':<25}: {f1_w:.4f}")
print(f"{'Cohen Kappa':<25}: {kappa:.4f}")
print(f"{'ROC-AUC':<25}: {auc:.4f}")
print()
print(classification_report(all_labels, all_preds,
                             target_names=class_names, zero_division=0))

## 10  Visualize Training History and Results

In [ ]:
from src.utils import smooth

epochs_ran = len(history["train_loss"])
ep_range   = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(ep_range, history["train_loss"], alpha=0.3, color="tab:blue",  label="Train (raw)")
axes[0].plot(ep_range, history["val_loss"],   alpha=0.3, color="tab:orange",label="Val (raw)")
axes[0].plot(ep_range, smooth(history["train_loss"]), color="tab:blue",   lw=2, label="Train (smooth)")
axes[0].plot(ep_range, smooth(history["val_loss"]),   color="tab:orange", lw=2, label="Val (smooth)")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(ep_range, history["train_acc"], alpha=0.3, color="tab:green", label="Train (raw)")
axes[1].plot(ep_range, history["val_acc"],   alpha=0.3, color="tab:red",   label="Val (raw)")
axes[1].plot(ep_range, smooth(history["train_acc"]), color="tab:green", lw=2, label="Train (smooth)")
axes[1].plot(ep_range, smooth(history["val_acc"]),   color="tab:red",   lw=2, label="Val (smooth)")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("Training History", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
from src.evaluate import plot_confusion_matrix, plot_roc, plot_precision_recall

# Inline display (also saved to results/)
cm   = confusion_matrix(all_labels, all_preds)
norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title, fmt in zip(
    axes,
    [cm, norm],
    ["Confusion Matrix (counts)", "Confusion Matrix (normalised)"],
    ["d", ".2f"],
):
    sns.heatmap(data, annot=True, fmt=fmt,
                xticklabels=class_names, yticklabels=class_names,
                cmap="Blues", linewidths=0.5, ax=ax)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.show()

In [ ]:
# ── ROC Curve ─────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(all_labels, all_probs[:, 1])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, lw=2, color="steelblue", label=f"ROC (AUC = {auc:.4f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="Random")
axes[0].fill_between(fpr, tpr, alpha=0.1, color="steelblue")
axes[0].set_xlabel("False Positive Rate (FPR)")
axes[0].set_ylabel("True Positive Rate (TPR)")
axes[0].set_title("ROC Curve")
axes[0].legend(); axes[0].grid(alpha=0.3)

# ── Precision-Recall Curve ────────────────────────────────────────────────────
prec, rec, _ = precision_recall_curve(all_labels, all_probs[:, 1])
ap = average_precision_score(all_labels, all_probs[:, 1])

axes[1].plot(rec, prec, lw=2, color="darkorange", label=f"PR (AP = {ap:.4f})")
axes[1].fill_between(rec, prec, alpha=0.1, color="darkorange")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 11  Real-Time Traffic Classification

Simulate a streaming environment where individual network-flow records arrive sequentially.
A circular buffer accumulates `SEQ_LEN` flows before the model fires a prediction.

In [ ]:
from collections import deque

class RealTimeDetector:
    """
    Wraps the trained LSTM for streaming inference.

    Usage
    -----
    detector = RealTimeDetector(model, scaler, seq_len=20, device=DEVICE)
    label, confidence = detector.ingest(raw_flow_vector)
    """

    def __init__(self, model, scaler, seq_len: int, class_names: list, device):
        self.model       = model.eval()
        self.scaler      = scaler
        self.seq_len     = seq_len
        self.class_names = class_names
        self.device      = device
        self.buffer      = deque(maxlen=seq_len)

    @torch.no_grad()
    def ingest(self, raw_flow: np.ndarray):
        """
        Accept a single flow feature vector (1-D, n_features).
        Returns (label_str, confidence) once the buffer is full, else None.
        """
        scaled = self.scaler.transform(raw_flow.reshape(1, -1))[0]
        self.buffer.append(scaled)

        if len(self.buffer) < self.seq_len:
            return None, None

        seq    = np.stack(list(self.buffer), axis=0)           # (T, F)
        tensor = torch.tensor(seq[None], dtype=torch.float32).to(self.device)
        probs  = torch.softmax(self.model(tensor), dim=1).cpu().numpy()[0]
        cls_id = probs.argmax()
        return self.class_names[cls_id], float(probs[cls_id])


# ── Instantiate ───────────────────────────────────────────────────────────────
detector = RealTimeDetector(model, scaler, SEQ_LEN, class_names, DEVICE)

# ── Simulate 60 incoming flows (mix benign + DDoS from test set) ──────────────
N_SIM        = 60
sim_indices  = np.random.choice(len(X_test), N_SIM, replace=False)
sim_flows    = X_test[sim_indices]           # raw (unscaled) – scaler applied inside
sim_truth    = y_test[sim_indices]

results = []
for i, (flow, truth) in enumerate(zip(sim_flows, sim_truth)):
    label, conf = detector.ingest(flow)
    if label is not None:
        results.append({
            "tick":       i,
            "predicted":  label,
            "confidence": conf,
            "true_label": class_names[truth],
            "correct":    label == class_names[truth],
        })

df_sim = pd.DataFrame(results)
print(df_sim.tail(20).to_string(index=False))
print(f"\nSimulation accuracy: {df_sim['correct'].mean():.4f}")

In [ ]:
# ── Real-time confidence plot ──────────────────────────────────────────────────
colors = ["#d62728" if r["predicted"] == "DDoS" else "#1f77b4"
          for _, r in df_sim.iterrows()]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(df_sim["tick"], df_sim["confidence"], color=colors, alpha=0.85, width=0.7)
ax.axhline(0.5, color="black", lw=1, ls="--", label="Decision boundary (0.5)")
ax.set_xlabel("Simulation tick")
ax.set_ylabel("Model confidence")
ax.set_title("Real-Time DDoS Detection — Confidence per Flow Sequence\n"
             "(Red = DDoS prediction, Blue = Benign prediction)")
ax.set_ylim(0, 1.05)
ax.legend()

# Annotate misclassifications
for _, row in df_sim[~df_sim["correct"]].iterrows():
    ax.annotate("✗", xy=(row["tick"], row["confidence"] + 0.02),
                ha="center", color="black", fontsize=10)

plt.tight_layout()
plt.show()

# ── Attention weights for a sample DDoS sequence ──────────────────────────────
ddos_idx = np.where(y_test_s == 1)[0][:1]
if len(ddos_idx):
    sample_seq = torch.tensor(X_test_s[ddos_idx], dtype=torch.float32).to(DEVICE)
    model.eval()
    with torch.no_grad():
        _, attn = model.forward_with_attention(sample_seq)
    attn = attn.cpu().numpy()[0]

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.bar(range(len(attn)), attn, color="mediumseagreen", alpha=0.85)
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Attention weight α")
    ax.set_title("BiLSTM Temporal Attention — DDoS Sample\n(which timesteps the model focuses on)")
    plt.tight_layout(); plt.show()